# COVID PNAD - TECH CHALLENGE

A pandemia da COVID-19 representou um dos maiores desafios recentes para os sistemas de saúde em todo o mundo, exigindo respostas rápidas, baseadas em dados e com forte capacidade analítica. Nesse contexto, compreender o comportamento da população, os padrões de sintomas e os impactos socioeconômicos torna-se essencial para antecipar cenários e estruturar estratégias mais eficazes em eventuais novos surtos.

Com esse objetivo, este projeto utiliza como base a pesquisa **PNAD COVID-19**, desenvolvida pelo IBGE, que fornece um conjunto robusto e confiável de dados sobre os efeitos da pandemia no Brasil. A escolha dessa base se justifica pela sua abrangência nacional, riqueza de variáveis e relevância estatística.

Para garantir foco analítico e eficiência na construção das análises, foi realizada uma seleção estratégica de até 20 questionamentos da pesquisa, contemplando três grandes dimensões fundamentais:

- **Características clínicas**: sintomas reportados pela população e indicadores relacionados à testagem da COVID-19;
- **Características populacionais**: aspectos demográficos como idade, sexo e localização geográfica;
- **Características econômicas**: impactos da pandemia na atividade de negócio ou ramo em que o a população trabalha;

Além disso, foram considerados dados de um recorte temporal de três meses (Julho, Agosto e Setembro de 2020), sem comprometer a performance e a escalabilidade da solução em ambiente de nuvem.

A arquitetura do projeto foi estruturada seguindo boas práticas de engenharia de dados, com organização em medalhão (camadas Bronze, Silver e Gold), garantindo qualidade, rastreabilidade e facilidade de consumo analítico.

Ao longo deste notebook, serão apresentadas as etapas de preparação dos dados, as transformações realizadas, as análises exploratórias e os principais insights obtidos. Por fim, serão propostas recomendações estratégicas para o hospital, com foco em prevenção, alocação de recursos e resposta a possíveis novos surtos da doença.

O objetivo central é transformar dados em informação relevante, apoiando decisões mais assertivas e contribuindo para um sistema de saúde mais preparado e resiliente.


####  Importando bibliotecas básicas


In [1]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

## Inicializando contexto do Glue e Spark

print("Etapa 01 - Contexto Glue e Spark")

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import create_map, lit
from itertools import chain
from pyspark.sql.functions import col, when, coalesce, lit, sum
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

#### Criação do DynamicFrame proveniente da tabela na AWS Glue Data Catalog e do seu respectivo schema


In [ ]:
## Leitura das tabelas do Glue Data Catalog

print("Etapa 02 - Leitura das tabelas do Glue Data Catalog")

dy_bronze = glueContext.create_dynamic_frame.from_catalog(database='default', table_name='covid_bronze')


#### Example: Convertendo o DynamicFrame para o Spark DataFrame e apresentando uma amostra da base de dados


In [3]:
## Conversão DynamicFrame -> Spark DataFrame

print("Etapa 03 - Conversão DynamicFrame -> Spark DataFrame")

df_covid = dy_bronze.toDF()
df_covid.printSchema()

#### Lendo os arquivos da camada bronze no bucket S3

In [4]:
## Leitura da camada bronze

print("Etapa 04 - Lendo os arquivos da camada bronze no bucket S3")

df_covid = spark.read.parquet("s3://lab-727776330445/data-output/bronze/Ano=2020/")

#### Tratamento dos dados da camada bronze e geração da camada silver
Definindo as interações dos dados através do dicionário e dos mapas conforme os módulos das perguntas 

In [5]:
## Tratamento e conversão dos dados - camada bronze -> Silver

print("Etapa 05 - Tratamento dos dados da camada bronze e geração da camada silver")


# UF
dict_uf = {
    11:"Rondônia",12:"Acre",13:"Amazonas",14:"Roraima",15:"Pará",
    16:"Amapá",17:"Tocantins",21:"Maranhão",22:"Piauí",23:"Ceará",
    24:"Rio Grande do Norte",25:"Paraíba",26:"Pernambuco",27:"Alagoas",
    28:"Sergipe",29:"Bahia",31:"Minas Gerais",32:"Espírito Santo",
    33:"Rio de Janeiro",35:"São Paulo",41:"Paraná",42:"Santa Catarina",
    43:"Rio Grande do Sul",50:"Mato Grosso do Sul",51:"Mato Grosso",
    52:"Goiás",53:"Distrito Federal"
}

map_uf = create_map([lit(x) for x in chain(*dict_uf.items())])


# Sexo
dict_sexo = {1:"Homem", 2:"Mulher"}
map_sexo = create_map([lit(x) for x in chain(*dict_sexo.items())])


# Sim/Não
dict_binario = {1:"Sim", 2:"Não"}
map_binario = create_map([lit(x) for x in chain(*dict_binario.items())])


# Sim/Não/Não sabe
dict_sintomas = {1:"Sim", 2:"Não", 3:"Não sabe"}
map_sintomas = create_map([lit(x) for x in chain(*dict_sintomas.items())])


# Resultado exames
dict_resultado = {
    1:"Positivo", 2:"Negativo",
    3:"Inconclusivo", 4:"Resultado pendente"
}
map_resultado = create_map([lit(x) for x in chain(*dict_resultado.items())])


# Internação
dict_internacao = {
    1:"Sim", 2:"Não", 3:"Não foi atendido"
}
map_internacao = create_map([lit(x) for x in chain(*dict_internacao.items())])


# Restrição social
dict_restricao = {
    1:"Vida normal",
    2:"Redução parcial",
    3:"Saiu apenas para necessidades",
    4:"Isolamento rigoroso"
}
map_restricao = create_map([lit(x) for x in chain(*dict_restricao.items())])


# Atividade econômica
dict_atividade = {
    1:"Agropecuária",2:"Extração",3:"Indústria",4:"Utilidades",
    5:"Construção",6:"Comércio",7:"Reparação veículos",
    8:"Transporte passageiros",9:"Transporte carga",
    10:"Logística",11:"Hospedagem",12:"Alimentação",
    13:"Informação e comunicação",14:"Financeiro",
    15:"Imobiliário",16:"Profissional técnico",
    17:"Serviços operacionais",18:"Administração pública",
    19:"Educação",20:"Saúde",21:"Organizações",
    22:"Atividades artísticas",23:"Serviços pessoais",
    24:"Serviço doméstico",25:"Outros"
}
map_atividade = create_map([lit(x) for x in chain(*dict_atividade.items())])

#### Tratamento dos dados da camada bronze e geração da camada silver
Renomeando as colunas e tratando os valores nulos

In [6]:
DEFAULT_VALUE = "Não respondido/ignorado"

df_silver = (
    df_covid
    # Básico
    .withColumn("estado", coalesce(map_uf[col("UF")], lit(DEFAULT_VALUE)))
    .withColumn("idade", col("A002"))
    .withColumn("sexo", coalesce(map_sexo[col("A003")], lit(DEFAULT_VALUE)))

    # Sintomas
    .withColumn("febre", coalesce(map_sintomas[col("B0011")], lit(DEFAULT_VALUE)))
    .withColumn("tosse", coalesce(map_sintomas[col("B0012")], lit(DEFAULT_VALUE)))
    .withColumn("dor_garganta", coalesce(map_sintomas[col("B0013")], lit(DEFAULT_VALUE)))
    .withColumn("dificuldade_respirar", coalesce(map_sintomas[col("B0014")], lit(DEFAULT_VALUE)))
    .withColumn("dor_cabeca", coalesce(map_sintomas[col("B0015")], lit(DEFAULT_VALUE)))
    .withColumn("dor_peito", coalesce(map_sintomas[col("B0016")], lit(DEFAULT_VALUE)))
    .withColumn("nausea", coalesce(map_sintomas[col("B0017")], lit(DEFAULT_VALUE)))
    .withColumn("nariz_entupido", coalesce(map_sintomas[col("B0018")], lit(DEFAULT_VALUE)))
    .withColumn("fadiga", coalesce(map_sintomas[col("B0019")], lit(DEFAULT_VALUE)))
    .withColumn("dor_olhos", coalesce(map_sintomas[col("B00110")], lit(DEFAULT_VALUE)))
    .withColumn("perda_olfato_paladar", coalesce(map_sintomas[col("B00111")], lit(DEFAULT_VALUE)))
    .withColumn("dor_muscular", coalesce(map_sintomas[col("B00112")], lit(DEFAULT_VALUE)))
    .withColumn("diarreia", coalesce(map_sintomas[col("B00113")], lit(DEFAULT_VALUE)))

    # Atendimento
    .withColumn("procurou_atendimento", coalesce(map_binario[col("B002")], lit(DEFAULT_VALUE)))
    .withColumn("atendimento_ubs", coalesce(map_binario[col("B0041")], lit(DEFAULT_VALUE)))
    .withColumn("atendimento_upa", coalesce(map_binario[col("B0042")], lit(DEFAULT_VALUE)))
    .withColumn("atendimento_hospital_sus", coalesce(map_binario[col("B0043")], lit(DEFAULT_VALUE)))
    .withColumn("atendimento_privado", coalesce(map_binario[col("B0044")], lit(DEFAULT_VALUE)))
    .withColumn("atendimento_ps_privado", coalesce(map_binario[col("B0045")], lit(DEFAULT_VALUE)))
    .withColumn("atendimento_hospital_privado", coalesce(map_binario[col("B0046")], lit(DEFAULT_VALUE)))

    # Internação
    .withColumn("internacao", coalesce(map_internacao[col("B005")], lit(DEFAULT_VALUE)))
    .withColumn("uso_ventilador", coalesce(map_binario[col("B006")], lit(DEFAULT_VALUE)))

    # Plano e teste
    .withColumn("plano_saude", coalesce(map_binario[col("B007")], lit(DEFAULT_VALUE)))
    .withColumn("fez_teste_covid", coalesce(map_binario[col("B008")], lit(DEFAULT_VALUE)))

    # Testes
    .withColumn("teste_swab", coalesce(map_binario[col("B009A")], lit(DEFAULT_VALUE)))
    .withColumn("resultado_swab", coalesce(map_resultado[col("B009B")], lit(DEFAULT_VALUE)))
    .withColumn("teste_dedo", coalesce(map_binario[col("B009C")], lit(DEFAULT_VALUE)))
    .withColumn("resultado_dedo", coalesce(map_resultado[col("B009D")], lit(DEFAULT_VALUE)))
    .withColumn("teste_sangue", coalesce(map_binario[col("B009E")], lit(DEFAULT_VALUE)))
    .withColumn("resultado_sangue", coalesce(map_resultado[col("B009F")], lit(DEFAULT_VALUE)))

    # Comorbidades
    .withColumn("diabetes", coalesce(map_binario[col("B0101")], lit(DEFAULT_VALUE)))
    .withColumn("hipertensao", coalesce(map_binario[col("B0102")], lit(DEFAULT_VALUE)))
    .withColumn("doenca_respiratoria", coalesce(map_binario[col("B0103")], lit(DEFAULT_VALUE)))
    .withColumn("doenca_cardiaca", coalesce(map_binario[col("B0104")], lit(DEFAULT_VALUE)))
    .withColumn("depressao", coalesce(map_binario[col("B0105")], lit(DEFAULT_VALUE)))
    .withColumn("cancer", coalesce(map_binario[col("B0106")], lit(DEFAULT_VALUE)))

    # Comportamento
    .withColumn("nivel_isolamento", coalesce(map_restricao[col("B011")], lit(DEFAULT_VALUE)))

    # Profissão
    .withColumn("atividade_economica", coalesce(map_atividade[col("C007D")], lit(DEFAULT_VALUE)))

    # Itens casa
    .withColumn("sabao", coalesce(map_sintomas[col("F002A1")], lit(DEFAULT_VALUE)))
    .withColumn("alcool", coalesce(map_sintomas[col("F002A2")], lit(DEFAULT_VALUE)))
    .withColumn("mascara", coalesce(map_sintomas[col("F002A3")], lit(DEFAULT_VALUE)))
    .withColumn("luva", coalesce(map_sintomas[col("F002A4")], lit(DEFAULT_VALUE)))
    .withColumn("desinfetante", coalesce(map_sintomas[col("F002A5")], lit(DEFAULT_VALUE)))
)

#### Convertendo a idade para inteiro

In [11]:
df_silver = df_silver.withColumn("idade", col("idade").cast("int"))

#### Elaborando a query do dataframe final
Selecionando apenas as colunas desejadas

In [7]:
## Selecionando apenas as colunas desejadas para a camada silver

print("Etapa 06 - Elaborando a query do df silver")

df_silver = df_silver.select(
    "estado",
    "idade",
    "sexo",
    "febre",
    "tosse",
    "dor_garganta",
    "dificuldade_respirar",
    "dor_cabeca",
    "dor_peito",
    "nausea",
    "nariz_entupido",
    "fadiga",
    "dor_olhos",
    "perda_olfato_paladar",
    "dor_muscular",
    "diarreia",
    "procurou_atendimento",
    "atendimento_ubs",
    "atendimento_upa",
    "atendimento_hospital_sus",
    "atendimento_privado",
    "atendimento_ps_privado",
    "atendimento_hospital_privado",
    "internacao",
    "uso_ventilador",
    "fez_teste_covid",
    "teste_swab",
    "resultado_swab",
    "teste_dedo",
    "resultado_dedo",
    "teste_sangue",
    "resultado_sangue",
    "diabetes",
    "hipertensao",
    "doenca_respiratoria",
    "doenca_cardiaca",
    "depressao",
    "cancer",
    "nivel_isolamento",
    "atividade_economica",
    "plano_saude",
    "sabao",
    "alcool",
    "mascara",
    "luva",
    "desinfetante"
)

In [8]:
## Verificando a quantidade de colunas do dataframe

print("Etapa 07 - Verificando a quantidade de colunas do dataframe")

print(len(df_silver.columns))

### Verificando a existência de dados nulos após tratamento da silver

In [9]:
## Verificando a existência de dados nulos

print("etapa 08 - Verificando a existência de dados nulos após tratamento")

null_counts = df_silver.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_silver.columns
])

# transforma em formato linha/coluna pra ordenar
from pyspark.sql.functions import expr

null_counts.selectExpr(
    "stack({0}, {1}) as (coluna, nulos)".format(
        len(df_silver.columns),
        ", ".join([f"'{c}', {c}" for c in df_silver.columns])
    )
).orderBy(col("nulos").desc()).show(len(df_silver.columns), truncate=False)

### Apontando a tabela silver para a pasta data-output/silver/ no bucket do S3

In [16]:
## Apontando a tabela silver para o bucket S3

print("Etapa 09 - Apontando a tabela silver para o bucket S3")

df_silver.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", "s3://lab-727776330445/data-output/silver/") \
    .saveAsTable("default.covid_silver")

In [12]:
df_silver.printSchema()

### Tratamento dos dados da camada gold

In [17]:
## Tratamento e conversão dos dados - camada Silver -> Gold

print("Etapa 10 - Tratamento dos dados da camada gold")


df_gold = (
    df_silver


    ## FEATURE ENGINEERING

    .withColumn(
        "qtd_sintomas",
        (
            (col("febre") == "Sim").cast("int") +
            (col("tosse") == "Sim").cast("int") +
            (col("dor_garganta") == "Sim").cast("int") +
            (col("dificuldade_respirar") == "Sim").cast("int") +
            (col("dor_cabeca") == "Sim").cast("int") +
            (col("dor_peito") == "Sim").cast("int") +
            (col("nausea") == "Sim").cast("int") +
            (col("nariz_entupido") == "Sim").cast("int") +
            (col("fadiga") == "Sim").cast("int") +
            (col("dor_olhos") == "Sim").cast("int") +
            (col("perda_olfato_paladar") == "Sim").cast("int") +
            (col("dor_muscular") == "Sim").cast("int") +
            (col("diarreia") == "Sim").cast("int")
        )
    )
    .withColumn(
        "tem_comorbidade",
        when(
            (col("diabetes") == "Sim") |
            (col("hipertensao") == "Sim") |
            (col("doenca_respiratoria") == "Sim") |
            (col("doenca_cardiaca") == "Sim") |
            (col("cancer") == "Sim"),
            "Sim"
        ).otherwise("Não")
    )
    .withColumn(
        "teve_atendimento",
        when(
            (col("atendimento_ubs") == "Sim") |
            (col("atendimento_upa") == "Sim") |
            (col("atendimento_hospital_sus") == "Sim") |
            (col("atendimento_privado") == "Sim") |
            (col("atendimento_ps_privado") == "Sim") |
            (col("atendimento_hospital_privado") == "Sim"),
            "Sim"
        ).otherwise("Não")
    )
    .withColumn(
    "status_teste",
    when(
        ### Se qualquer um for positivo → Positivo
        (col("resultado_swab") == "Positivo") |
        (col("resultado_dedo") == "Positivo") |
        (col("resultado_sangue") == "Positivo"),
        "Positivo"
    ).when(
        ### Nenhum positivo E pelo menos um negativo → Negativo
        (
            (col("resultado_swab") != "Positivo") &
            (col("resultado_dedo") != "Positivo") &
            (col("resultado_sangue") != "Positivo")
        ) &
        (
            (col("resultado_swab") == "Negativo") |
            (col("resultado_dedo") == "Negativo") |
            (col("resultado_sangue") == "Negativo")
        ),
        "Negativo"
    ).otherwise("Indefinido")
    )
    .withColumn(
        "score_isolamento",
        when(col("nivel_isolamento") == "Vida normal", 1)
        .when(col("nivel_isolamento") == "Redução parcial", 2)
        .when(col("nivel_isolamento") == "Saiu apenas para necessidades", 3)
        .when(col("nivel_isolamento") == "Isolamento rigoroso", 4)
    )
    .withColumn(
        "caso_grave",
        when(
            (col("internacao") == "Sim") |
            (col("uso_ventilador") == "Sim"),
            "Sim"
        ).otherwise("Não")
    )
    .withColumn(
        "indice_risco",
        when(
            (col("idade") >= 60) & (col("tem_comorbidade") == "Sim"),
            "Alto"
        ).when(
            (col("qtd_sintomas") >= 3),
            "Médio"
        ).otherwise("Baixo")
    )
    .withColumn(
        "regiao",
        when(col("estado").isin(
            "Acre","Amapá","Amazonas","Pará","Rondônia","Roraima","Tocantins"
        ), "Norte")
        .when(col("estado").isin(
            "Alagoas","Bahia","Ceará","Maranhão","Paraíba",
            "Pernambuco","Piauí","Rio Grande do Norte","Sergipe"
        ), "Nordeste")
        .when(col("estado").isin(
            "Goiás","Mato Grosso","Mato Grosso do Sul","Distrito Federal"
        ), "Centro-Oeste")
        .when(col("estado").isin(
            "Espírito Santo","Minas Gerais","Rio de Janeiro","São Paulo"
        ), "Sudeste")
        .when(col("estado").isin(
            "Paraná","Rio Grande do Sul","Santa Catarina"
        ), "Sul")
        .otherwise("Não respondido/ignorado")
    )
    .withColumn(
        "faixa_etaria",
        when(col("idade") <= 17, "0-17")
        .when(col("idade") <= 39, "18-39")
        .when(col("idade") <= 59, "40-59")
        .otherwise("60+")
    )
    .withColumn(
        "nivel_severidade",
        when(col("qtd_sintomas") > 5, "Alta")
        .when(col("qtd_sintomas") >= 3, "Média")
        .otherwise("Baixa")
    )
    .withColumn(
        "qtd_prevencao",
    (
        (col("mascara") == "Sim").cast("int") +
        (col("alcool") == "Sim").cast("int") +
        (col("sabao") == "Sim").cast("int") +
        (col("luva") == "Sim").cast("int") +
        (col("desinfetante") == "Sim").cast("int")
    )
    .withColumn(
        "kit_prevencao",
        when(col("qtd_prevencao") >= 2, "Completo")
        .otherwise("Incompleto")
    )
    .withColumn(
        "perfil_risco_comportamental",
        when(
            (col("score_isolamento") <= 2) &
            (col("kit_prevencao") == "Incompleto"),
            "Alto risco"
        ).when(
            (col("score_isolamento") == 3),
            "Médio risco"
        ).otherwise("Baixo risco")
    )
    
    .select(

               "estado",
                "idade",
                "sexo",
                "febre",
                "tosse",
                "dor_garganta",
                "dificuldade_respirar",
                "dor_cabeca",
                "dor_peito",
                "nausea",
                "nariz_entupido",
                "fadiga","dor_olhos",
                "perda_olfato_paladar",
                "dor_muscular",
                "diarreia",
                "procurou_atendimento",
                "atendimento_ubs",
                "atendimento_upa",
                "atendimento_hospital_sus",
                "atendimento_privado",
                "atendimento_ps_privado",
                "atendimento_hospital_privado",
                "internacao",
                "uso_ventilador",
                "fez_teste_covid",
                "teste_swab",
                "resultado_swab",
                "teste_dedo",
                "resultado_dedo",
                "teste_sangue",
                "resultado_sangue",
                "diabetes",
                "hipertensao",
                "doenca_respiratoria",
                "doenca_cardiaca",
                "depressao",
                "cancer",
                "nivel_isolamento",
                "atividade_economica",
                "plano_saude",
                "sabao",
                "alcool",
                "mascara",
                "luva",
                "desinfetante",

        ## GOLD FEATURES

                "qtd_sintomas",
                "tem_comorbidade",
                "teve_atendimento",
                "status_teste",
                "score_isolamento",
                "caso_grave",
                "indice_risco",
                "regiao",
                "faixa_etaria",
                "nivel_severidade",
                "kit_prevencao",
                "perfil_risco_comportamental"
                )
            )

In [19]:
df_gold.printSchema()

In [18]:
df_gold.show(5, truncate=False)

### Apontando a tabela Gold para a pasta data-output/gold/ no bucket do S3

In [22]:
## Apontando a tabela gold para o bucket S3

print("Etapa 11 - Apontando a tabela gold para o bucket S3")


df_gold.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", "s3://lab-727776330445/data-output/gold/") \
    .saveAsTable("default.covid_gold")